In [3]:
# 外部资源集成
!pip install langchain_community

# dotenv管理密钥，加载环境变量等
!pip install python-dotenv

!pip install langchain-classic

!pip install langchain

In [4]:
from dotenv import load_dotenv
##  一个售前和售后的 langchain  LLMRouterChain 模版

from langchain_classic.chains.router import MultiPromptChain
from langchain_community.llms import Tongyi
from langchain_classic.chains import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser
)
from langchain_classic.chains.router.multi_prompt_prompt import (
    MULTI_PROMPT_ROUTER_TEMPLATE
)
import os

load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

# 售前咨询模板
presales_prompt_tpl = PromptTemplate.from_template(
    '你是一位专业的售前顾问，擅长产品介绍、方案推荐和商务咨询。'
    '你需要热情、专业地回答客户的产品咨询、价格询问、功能介绍等售前问题。'
    '请使用中文帮我解答下列售前咨询问题：\n{input}'
)

# 售后服务模板
aftersales_prompt_tpl = PromptTemplate.from_template(
    '你是一位耐心的售后服务专员，擅长解决客户的使用问题、技术支持和投诉处理。'
    '你需要耐心、细致地帮助客户解决产品使用中遇到的问题，提供技术支持和服务指导。'
    '请使用中文帮我解答下列售后服务问题：\n{input}'
)

# 创建模板信息列表
prompt_infos = [
    {
        'name': 'presales',
        'description': '用于处理售前咨询，包括产品介绍、价格询问、功能说明、方案推荐等',
        'prompt_template': presales_prompt_tpl,
    },
    {
        'name': 'aftersales',
        'description': '用于处理售后服务，包括使用问题、技术支持、故障排除、投诉处理等',
        'prompt_template': aftersales_prompt_tpl,
    },
]

llm = Tongyi(
    temperature=0.1,
)

# 生成键为模板名称、值为Chain的字典
destination_chains = {}
for p_info in prompt_infos:
    name = p_info['name']
    prompt = p_info['prompt_template']
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

# 将模板名称和模板描述通过MULTI_PROMPT_ROUTER_TEMPLATE生成模板
destinations = [f'{p["name"]}: {p["description"]}'
                for p in prompt_infos]
destinations_str = "\n".join(destinations)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=['input'],
    output_parser=RouterOutputParser(),
)
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

# 这里创建了一个default_chain
# 为了防止提的问题类型并没有包含在prompt_infos中
default_chain = ConversationChain(llm=llm, output_key='text')
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True,
)

# 测试售前咨询问题
print("=== 售前咨询测试 ===")
print(chain.run("你们的产品有什么功能？价格是多少？"))

print("\n=== 售后服务测试 ===")
print(chain.run("我的产品出现故障了，无法正常启动，该怎么办？"))

print("\n=== 其他问题测试 ===")
print(chain.run("今天天气怎么样？"))


C:\Users\LiuHuijun\AppData\Local\Temp\ipykernel_48696\1151766320.py:58: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)
C:\Users\LiuHuijun\AppData\Local\Temp\ipykernel_48696\1151766320.py:78: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  default_chain = ConversationChain(llm=llm, output_key='text')
D:\miniconda3\envs\week04\Lib\site-packages\pydantic\main.py:250: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
C:\Users\LiuHuijun\AppData\Local\Temp\ipykernel_48696\1151766320.py:79: LangChainDeprecationWarning: Pleas

=== 售前咨询测试 ===


> Entering new MultiPromptChain chain...


AuthenticationError: No api key provided. You can set by dashscope.api_key = your_api_key in code, or you can set it via environment variable DASHSCOPE_API_KEY= your_api_key. You can store your api key to a file, and use dashscope.api_key_file_path=api_key_file_path in code, or you can set api key file path via environment variable DASHSCOPE_API_KEY_FILE_PATH, You can call save_api_key to api_key_file_path or default path(~/.dashscope/api_key).

In [11]:
# 整合链的语法

from langchain_community.llms import Tongyi
from langchain_classic.chains import ConversationChain
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 初始化LLM
llm = Tongyi(temperature=0.1)

# 售前咨询链 - 使用新式语法
presales_prompt = PromptTemplate.from_template(
    '你是一位专业的售前顾问，擅长产品介绍、方案推荐和商务咨询。'
    '你需要热情、专业地回答客户的产品咨询、价格询问、功能介绍等售前问题。'
    '请使用中文帮我解答下列售前咨询问题：\n{input}'
)
presales_chain = presales_prompt | llm | StrOutputParser()

# 售后服务链 - 使用新式语法
aftersales_prompt = PromptTemplate.from_template(
    '你是一位耐心的售后服务专员，擅长解决客户的使用问题、技术支持和投诉处理。'
    '你需要耐心、细致地帮助客户解决产品使用中遇到的问题，提供技术支持和服务指导。'
    '请使用中文帮我解答下列售后服务问题：\n{input}'
)
aftersales_chain = aftersales_prompt | llm | StrOutputParser()

# 意图识别链 - 使用新式语法
intent_prompt = PromptTemplate.from_template(
    """请分析以下用户问题的意图，判断是售前咨询还是售后服务：

售前咨询：产品介绍、功能说明、价格询问、方案推荐、购买咨询等
售后服务：使用问题、技术支持、故障排除、投诉处理、维修服务等

用户问题：{input}

请只回答"售前"或"售后"，不要添加其他内容。"""
)
intent_chain = intent_prompt | llm | StrOutputParser()

# 创建路由函数
def route_question(input_dict):
    question = input_dict["input"]
    intent = intent_chain.invoke({"input": question})
    
    print(f"识别意图: {intent.strip()}")
    
    if "售前" in intent:
        return presales_chain.invoke({"input": question})
    elif "售后" in intent:
        return aftersales_chain.invoke({"input": question})
    else:
        # 默认处理
        default_prompt = PromptTemplate.from_template(
            "我是一个智能助手，很高兴为您服务。请问有什么可以帮助您的吗？\n问题：{input}"
        )
        default_chain = default_prompt | llm | StrOutputParser()
        return default_chain.invoke({"input": question})

# 创建完整的路由链
router_chain = RunnablePassthrough() | RunnableLambda(route_question)

# 方法二：更简洁的条件路由实现
from langchain_core.runnables import RunnableBranch

# 创建条件判断函数
def is_presales(input_dict):
    intent = intent_chain.invoke(input_dict)
    return "售前" in intent

def is_aftersales(input_dict):
    intent = intent_chain.invoke(input_dict)
    return "售后" in intent

# 使用 RunnableBranch 创建条件路由
branch_chain = RunnableBranch(
    (is_presales, presales_chain),
    (is_aftersales, aftersales_chain),
    # 默认链
    PromptTemplate.from_template("我是智能助手，请问有什么可以帮助您的？\n{input}") | llm | StrOutputParser()
)

# 测试代码
if __name__ == "__main__":
    print("=== 方法一：自定义路由函数 ===")
    
    # 测试售前问题
    print("\n--- 售前咨询测试 ---")
    result1 = router_chain.invoke({"input": "你们的产品有什么功能？价格是多少？"})
    print(f"回答: {result1}")
    
    # 测试售后问题
    print("\n--- 售后服务测试 ---")
    result2 = router_chain.invoke({"input": "我的产品出现故障了，无法正常启动，该怎么办？"})
    print(f"回答: {result2}")
    
    print("\n=== 方法二：RunnableBranch 条件路由 ===")
    
    # 测试售前问题
    print("\n--- 售前咨询测试 ---")
    result3 = branch_chain.invoke({"input": "我想了解一下你们的服务套餐和收费标准"})
    print(f"回答: {result3}")
    
    # 测试售后问题
    print("\n--- 售后服务测试 ---")
    result4 = branch_chain.invoke({"input": "产品使用过程中遇到了错误提示，需要技术支持"})
    print(f"回答: {result4}")


=== 方法一：自定义路由函数 ===

--- 售前咨询测试 ---
识别意图: 售前
回答: 您好！非常感谢您的关注和咨询！😊  
作为专业的售前顾问，我很乐意为您清晰、全面地介绍我们的产品——不过需要先向您说明：我们提供的是**模块化、可定制的企业级智能解决方案**（如AI客服系统、智能工单平台、数据集成中台等），不同行业（如金融、电商、政务、教育）和不同规模客户（中小企业/大型集团）的实际需求差异较大，因此**产品功能组合与报价会基于具体业务场景动态配置**，而非“一刀切”的标准版。

为更精准地服务您，我先为您概括核心能力框架，并附上典型方案参考：

✅ **核心功能亮点**  
🔹 **智能交互层**：支持多渠道接入（微信/APP/网页/电话）、NLP语义理解、上下文对话记忆、7×24小时自动应答（准确率≥92%）；  
🔹 **业务协同层**：工单自动分派+SLA超时预警、知识库智能推荐、与CRM/ERP/OA系统一键对接（已预置50+主流系统API）；  
🔹 **数据决策层**：实时服务看板、客户情绪分析、话术优化建议、ROI效果归因报告（支持自定义KPI）；  
🔹 **安全合规保障**：等保三级认证、私有化部署选项、数据不出域、GDPR/《个人信息保护法》双合规。

💰 **关于价格**  
我们采用「按需订阅 + 弹性扩容」模式，无隐藏费用：  
• **轻量版（适合10人以内团队）**：¥3,800/月起，含基础AI客服+5个坐席+标准API对接；  
• **专业版（中型企业主力选择）**：¥12,800/月起，含全功能模块+定制知识库+专属实施顾问+季度优化服务；  
• **旗舰版（集团级/私有化部署）**：面议（含源码授权、信创适配、7×24驻场支持等）。  
✨ *注：首年签约享免费POC验证（15天真实业务场景测试），并赠送《行业最佳实践白皮书》。*

📌 为了给您推荐最匹配的方案，能否请您简单告知：  
① 您所在的行业及主要业务场景？（例如：电商售后咨询、银行理财问答、政府热线分流）  
② 当前服务渠道有哪些？日均咨询量大概多少？  
③ 是否已有CRM或其它业务系统？希望优先解决什么痛点？（如响应慢、人力成本高、满意度低等）

我将立即为您生成一份**个性化方案摘要+3年TCO对比表**，并安排技术专家为您做1对1演示。

## EmbeddingRouterChain

不仅可以使用 LLMRouteChain 来智能选择合适的处理链，还可以采用 EmbeddingRouterChain，该组件通过计算各 Chain 描述与用户问题之间的语义相关性，实现更精准的路由决策。


In [1]:
!pip install chromadb

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/23.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.1 MB ? eta -:--:--
   --------------------

In [6]:
from langchain_community.vectorstores import Chroma    # # pip install chroma
from langchain_community.embeddings import DashScopeEmbeddings # pip install dashscope
from langchain_classic.chains import LLMRouterChain, MultiPromptChain
from langchain_core.language_models import BaseLLM
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Tongyi  # 或你使用的 LLM
import os

# 1. 定义任务名称与描述
names_and_descriptions = [
    ("physics", ["用于解答物理相关问题，例如力学、电磁学等"]), 
    ("math", ["用于解答数学相关问题，例如代数、几何、微积分等"]), 
]

# 2. 使用通义千问的 Embedding 模型
embeddings = DashScopeEmbeddings(model="text-embedding-v2")

# 3. 构建向量数据库（用于路由匹配）
descriptions = []
names = []
for name, desc_list in names_and_descriptions:
    for desc in desc_list:
        descriptions.append(desc)
        names.append(name)

# 创建 Chroma 向量库
vectorstore = Chroma(embedding_function=embeddings)
# 批量添加文档
vectorstore.add_texts(texts=descriptions, metadatas=[{"name": name} for name in names])

# 4. 自定义 Embedding 路由链（LangChain 没有直接 from_names_and_descriptions）
def get_relevant_chain_name(question: str) -> str:
    docs = vectorstore.similarity_search(question, k=1)
    return docs[0].metadata["name"]

# 5. 定义各个目标链的 prompt 和 LLM
llm = Tongyi(model_name="qwen-plus", temperature=0.1)  # 可替换为你用的 LLM

physics_prompt = PromptTemplate(
    template="你是一个物理专家，请回答以下问题：\n{input}",
    input_variables=["input"]
)
math_prompt = PromptTemplate(
    template="你是一个数学专家，请回答以下问题：\n{input}",
    input_variables=["input"]
)

destination_chains = {
    "physics": physics_prompt | llm,
    "math": math_prompt | llm,
}

default_chain = PromptTemplate.from_template("请回答以下问题：{input}") | llm

# 6. 定义运行逻辑（模拟 MultiPromptChain）
def run_router_chain(question: str):
    chain_name = get_relevant_chain_name(question)
    print(f"路由到: {chain_name}")
    if chain_name in destination_chains:
        return destination_chains[chain_name].invoke({"input": question})
    else:
        return default_chain.invoke({"input": question})

# 7. 测试
result = run_router_chain("牛顿第一定律是什么？")
print(result)

路由到: physics
牛顿第一定律，又称**惯性定律**，其经典表述为：

> **任何物体在不受外力作用（或所受合外力为零）时，总保持静止状态或匀速直线运动状态。**

### 关键要点解析：
1. **揭示惯性本质**：  
   定律指出，物体具有保持其原有运动状态（静止或匀速直线运动）的内在属性，这种属性称为**惯性**。惯性的大小由物体的**质量**决定（质量越大，惯性越大）。

2. **定义惯性参考系**：  
   牛顿第一定律不仅描述物体行为，更**隐含地定义了惯性参考系**——即在该参考系中，若物体不受力（或合力为零），其加速度为零。只有在惯性系中，牛顿定律才严格成立。地面参考系在多数宏观低速情形下可近似视为惯性系；而加速运动的车厢、旋转地球等非惯性系中，需引入惯性力才能形式上维持定律适用。

3. **否定“力是维持运动的原因”这一亚里士多德式错误观念**：  
   历史上，人们曾认为运动需要持续受力才能维持。牛顿第一定律明确指出：**力不是维持速度的原因，而是改变运动状态（即产生加速度）的原因**。物体之所以停下（如滑动的木块），是因为存在摩擦力等外力，而非“自然趋向静止”。

4. **数学表达**：  
   当 \(\sum \vec{F} = 0\) 时，\(\vec{a} = \frac{d\vec{v}}{dt} = 0\)，故 \(\vec{v} = \text{常矢量}\)（包括 \(\vec{v} = \vec{0}\) 的特例）。

✅ 补充说明：该定律是牛顿力学体系的基石，为第二定律（\( \vec{F} = m\vec{a} \)）提供了前提和参照框架——它确立了“无外力则无加速度”的基准情形。

如需进一步探讨其与相对论的关系（例如在狭义相对论中，惯性定律在所有惯性系中依然成立，但时空结构被推广为闵可夫斯基时空），欢迎继续提问！
